In [1]:
import pypowsybl as pp
import pypowsybl.network as pn
import pypowsybl.loadflow as lf
from OMPython import ModelicaSystem
import pandas as pd
import numpy as np
from dynawo_notebooks.Scripts.core.grid_manager import GridManager
from dynawo_notebooks.Scripts.core.modelica_wrapper import ModelicaWrapper
from dynawo_notebooks.Scripts.core.orchestrator import Orchestrator

In [2]:
# Instantiate manager
gm = GridManager("MyBess_Scenario")

# 1. Create Buses
# Using 20kV voltage level
gm.create_bus("Bus_Grid", nominal_kv=20.0)
gm.create_bus("Bus_PCC", nominal_kv=20.0)

# 2. Add Source (External Grid)
# Thanks to reverse lookup, '1.02' is automatically converted to 20.4 kV
gm.add_source("GridSource", "Bus_Grid", v_pu=1.02)

# 3. Add "MyBess" Battery at the PCC
# Simulating a 5 MW discharge into the grid
gm.add_battery("MyBess", "Bus_PCC", p_mw=5.0, q_mvar=1.0)

# 4. Add Local Load at the PCC
gm.add_load("LocalLoad", "Bus_PCC", p_mw=3.5, q_mvar=0.5)

# 5. Connect with a line
gm.add_line("L1", "Bus_Grid", "Bus_PCC", r=0.5, x=1.2)

print("Grid configured successfully.")

Grid configured successfully.


In [8]:
# Model path
MO_FILE = "Models/MyBESS_static.mo"
MODEL_NAME = "MyBESS_static"

# Load Model
try:
    mod = ModelicaSystem(MO_FILE, MODEL_NAME)
    print("Modelica model loaded successfully.")
except Exception as e:
    print(f"Error loading Modelica (File exists?): {e}")
    # For demo purposes if you don't have the real file:
    print(">> DEMO MODE: Continuing load flow simulation without real Modelica backend.")

Error loading Modelica (File exists?): Error executing 'loadFile("/home/guiu/Projects/dynawo-notebooks/src/dynawo_notebooks/Models/MyBESS_static.mo")'
>> DEMO MODE: Continuing load flow simulation without real Modelica backend.


In [9]:
# Initialize orchestrator
orch = Orchestrator(gm, mod)

# --- MAPPING DEFINITION ---
# Connect PyPowSyBl names (left) with Modelica paths (right)

# Battery Mapping
orch.add_mapping("MyBess", "bessComponent")
# This will look for 'bessComponent.P_start', 'bessComponent.Q_start'

# PCC Mapping (Voltage and Angle to initialize the bus in Modelica)
orch.add_mapping("Bus_PCC", "pccBus")
# This will look for 'pccBus.v_start', 'pccBus.angle_start'

# Source Mapping
orch.add_mapping("Bus_Grid", "gridBus")

# --- EXECUTION ---
orch.sync()

NameError: name 'mod' is not defined

In [ ]:
import matplotlib.pyplot as plt

# Configure simulation
mod.setSimulationOptions()

# Simulate
mod.simulate()

# Retrieve results
# Assuming standard variable names in the model


# Plot
time = res
v_pcc = res[1]
p_bess = res[2]

plt.figure(figsize=(10, 6))

plt.subplot(2, 1, 1)
plt.plot(time, v_pcc)
plt.title("Voltage at PCC (Should be constant if Flat Start worked)")
plt.ylabel("Voltage (V)")
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(time, p_bess)
plt.title("Battery Active Power")
plt.ylabel("Power (W)")
plt.xlabel("Time (s)")
plt.grid(True)

plt.tight_layout()
plt.show()